# So sánh B0 và B1-P2 trên Validation — YOLOv8s (Dual-Mode: Local Mac & Kaggle GPU)

Notebook đánh giá và so sánh hai checkpoint đã huấn luyện:
- **B0 (YOLOv8s Baseline)**: `runs/B0_yolov8s_100ep_batch16/weights/best.pt`
- **B1 (YOLOv8s-P2)**: `runs/B1_yolov8s_p2_100ep/weights/best.pt`

Thực hiện trên cùng 608 ảnh validation (`data/splits/val.txt`), độ phân giải `imgsz=1024`.
Tự động nhận diện chạy trên **Local Mac (Apple MPS GPU)** hoặc **Kaggle GPU (Tesla T4/P100)**.


## 1. Môi trường và xác thực đầu vào
Cần `ultralytics`, `torch`, `numpy`, `pandas`, `matplotlib`, `PyYAML`.

In [4]:
# Cell 2 — Thiết lập môi trường, định vị Checkpoints & Chuẩn bị dữ liệu Validation
import hashlib, json, os, platform, shutil, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display, FileLink

IS_KAGGLE = Path('/kaggle').exists()
if IS_KAGGLE:
    import subprocess
    print('🚀 Đang chạy trên Kaggle GPU — Cài đặt môi trường...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'ray', '-y', '-q'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.3.40', '-q'])

from ultralytics import YOLO

CLASSES = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']
SIZE_BINS = ['small', 'medium', 'large']
IMGSZ, VAL_CONF, PREDICT_CONF, NMS_IOU, MATCH_IOU = 1024, 0.001, 0.25, 0.7, 0.50

if IS_KAGGLE:
    ROOT = Path('/kaggle/working')
    os.chdir(ROOT)
    input_root = Path('/kaggle/input')
    
    # Tìm dataset PKU
    pku_candidates = [p for p in input_root.rglob('PKU-Market-PCB(Data enhanced version)') if (p/'train/images').is_dir()]
    assert len(pku_candidates) >= 1, 'Không tìm thấy dataset PKU trong Kaggle input!'
    KAGGLE_IMG_ROOT = pku_candidates[0]
    alt_local_dirs = [KAGGLE_IMG_ROOT]
    
    # Tìm split val.txt từ bundle
    val_split_cands = list(input_root.rglob('val.txt'))
    assert len(val_split_cands) >= 1, 'Không tìm thấy file val.txt trong Kaggle input bundle!'
    VAL_TXT = val_split_cands[0]
    
    # Tìm checkpoints B0 và B1 trên Kaggle
    b0_pt, b1_pt = None, None
    for p in input_root.rglob('*.pt'):
        p_str = str(p).lower()
        if 'p2' in p_str and ('best' in p_str or 'b1' in p_str):
            b1_pt = p
        elif ('b0' in p_str or 'batch16' in p_str) and 'best' in p_str:
            b0_pt = p
            
    for z in input_root.rglob('*.zip'):
        z_str = str(z).lower()
        if 'b0' in z_str and b0_pt is None:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(ROOT / 'extracted_b0')
            cand = list((ROOT / 'extracted_b0').rglob('best.pt'))
            if cand: b0_pt = cand[0]
        elif ('b1' in z_str or 'p2' in z_str) and b1_pt is None:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(ROOT / 'extracted_b1')
            cand = list((ROOT / 'extracted_b1').rglob('best.pt'))
            if cand: b1_pt = cand[0]
            
    assert b0_pt is not None, 'Không tìm thấy checkpoint B0 trên Kaggle. Hãy Add Input dataset chứa kết quả B0.'
    assert b1_pt is not None, 'Không tìm thấy checkpoint B1-P2 trên Kaggle. Hãy Add Input dataset chứa kết quả B1.'
    
    CHECKPOINTS = {'B0 (YOLOv8s)': b0_pt, 'B1 (YOLOv8s-P2)': b1_pt}
    DEVICE = 0
    BATCH = 8
else:
    cwd = Path.cwd().resolve()
    ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'configs' / 'yolov8s-p2.yaml').is_file()), None)
    assert ROOT is not None, 'Mở notebook trong project PCB-DetectionImprove.'
    os.chdir(ROOT)
    VAL_TXT = ROOT / 'data/splits/val.txt'
    
    def find_checkpoint(cand_list):
        for p in cand_list:
            if p.is_file(): return p
        return cand_list[0]
        
    CHECKPOINTS = {
        'B0 (YOLOv8s)': find_checkpoint([
            ROOT / 'runs/B0_yolov8s_100ep_batch16/weights/best.pt',
            ROOT / 'experiences/B0_yolov8s_100ep/weights/best.pt'
        ]),
        'B1 (YOLOv8s-P2)': find_checkpoint([
            ROOT / 'runs/B1_yolov8s_p2_100ep/weights/best.pt',
            ROOT / 'experiences/B1_yolov8s_p2_100ep/weights/best.pt'
        ]),
    }
    alt_local_dirs = [
        Path('/Users/duytan/orca/projects/PCB-Detect/PKU-Market-PCB(Data enhanced version)'),
        ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)'
    ]
    if torch.cuda.is_available():
        DEVICE = 0; BATCH = 8; print('CUDA GPU:', torch.cuda.get_device_name(0))
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        DEVICE = 'mps'; BATCH = 4; print('Apple MPS (Apple Silicon GPU)')
    else:
        DEVICE = 'cpu'; BATCH = 1; print('CPU: đánh giá có thể lâu')

assert VAL_TXT.is_file(), f'Thiếu file split: {VAL_TXT}'
for name, path in CHECKPOINTS.items():
    assert path.is_file(), f'{name}: thiếu {path}'

LOCAL_PREFIX = 'data/raw/PKU-Market-PCB(Data enhanced version)/'
def resolve_image(line):
    p = Path(line)
    if p.is_file(): return p.resolve()
    if (ROOT / line).is_file(): return (ROOT / line).resolve()
    rel = line[len(LOCAL_PREFIX):] if line.startswith(LOCAL_PREFIX) else line
    for alt in alt_local_dirs:
        if (alt / rel).is_file(): return (alt / rel).resolve()
    return (ROOT / p).resolve()

def label_path(image):
    parts = list(image.parts)
    assert 'images' in parts, f'Không xác định được đường dẫn label: {image}'
    parts[parts.index('images')] = 'labels'
    return Path(*parts).with_suffix('.txt')

val_lines = [s.strip() for s in VAL_TXT.read_text().splitlines() if s.strip()]
assert len(val_lines) == 608, f'Val split phải có 608 ảnh, thấy {len(val_lines)}; dừng để kiểm tra.'
VAL_IMAGES = [resolve_image(s) for s in val_lines]
missing = [p for p in VAL_IMAGES if not p.is_file() or not label_path(p).is_file()]
assert not missing, f'Thiếu ảnh/label val, ví dụ: {missing[:3]}'

# Thiết lập thư mục output & file split với đường dẫn tuyệt đối
OUT = ROOT / 'evaluation/B0_B1_yolov8s_val'
OUT.mkdir(parents=True, exist_ok=True)
VAL_EVAL_SPLIT = OUT / 'val_eval_split.txt'
with open(VAL_EVAL_SPLIT, 'w', encoding='utf-8') as f:
    for p in VAL_IMAGES:
        f.write(str(p.resolve()) + '\n')

eval_yaml_data = {
    'path': str(ROOT),
    'train': str(VAL_EVAL_SPLIT.resolve()),
    'val': str(VAL_EVAL_SPLIT.resolve()),
    'nc': len(CLASSES),
    'names': {i: name for i, name in enumerate(CLASSES)}
}
EVAL_DATA_YAML = OUT / 'eval_val_data.yaml'
EVAL_DATA_YAML.write_text(yaml.safe_dump(eval_yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')

SIZE_DEVICE = DEVICE
SIZE_BATCH = BATCH
print('✅ Environment:', 'KAGGLE' if IS_KAGGLE else 'LOCAL MAC', '| Device:', DEVICE, '| Batch:', BATCH)
print('✅ Val images:', len(VAL_IMAGES), '| Val split SHA256:', hashlib.sha256(VAL_TXT.read_bytes()).hexdigest()[:8])
for n, p in CHECKPOINTS.items():
    print(f'   • {n:16s}: {p} ({round(p.stat().st_size / 1024**2, 2)} MiB)')


## 2. Chạy validation công bằng
Nạp từng checkpoint riêng, đánh giá cùng split và tham số. Các metric dưới đây được tính lại, không dùng số hardcode trong notebook khác.

In [ ]:
OUT = ROOT / 'evaluation/B0_B1_yolov8s_val'
OUT.mkdir(parents=True, exist_ok=True)
models, overall_rows, class_rows = {}, [], []
for name, checkpoint in CHECKPOINTS.items():
    print('\nValidation:', name)
    model = YOLO(str(checkpoint))
    t0 = time.time()
    result = model.val(data=str(EVAL_DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        conf=VAL_CONF, iou=NMS_IOU, max_det=300, plots=True, verbose=True,
        project=str(OUT / 'ultralytics_runs'), name=name.split()[0], exist_ok=True)
    box = result.box
    overall_rows.append({'model': name, 'images': len(VAL_IMAGES), 'imgsz': IMGSZ, 'mAP50': float(box.map50),
        'mAP50_95': float(box.map), 'precision': float(box.mp), 'recall': float(box.mr),
        'elapsed_seconds': time.time() - t0})
    observed = [int(x) for x in box.ap_class_index]
    for cid, cls in enumerate(CLASSES):
        j = observed.index(cid) if cid in observed else None
        p, r, a50, a = (float(box.p[j]), float(box.r[j]), float(box.ap50[j]), float(box.ap[j])) if j is not None else (0., 0., 0., 0.)
        class_rows.append({'model': name, 'class_id': cid, 'class': cls, 'precision': p, 'recall': r, 'mAP50': a50, 'mAP50_95': a})
    models[name] = model
    print('Elapsed minutes and metrics:', round((time.time() - t0) / 60, 2), 'min | mAP50:', float(box.map50), '| mAP50-95:', float(box.map))
    del result

overall = pd.DataFrame(overall_rows).set_index('model')
per_class = pd.DataFrame(class_rows)
display(overall.round(4))
display(per_class.pivot(index='class', columns='model', values='recall').reindex(CLASSES).round(4))



Validation: B0 (YOLOv8s)
Ultralytics 8.4.153 🚀 Python-3.13.5 torch-2.14.0 MPS (Apple M1 Pro)
Model summary (fused): 72 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1576.3±946.4 MB/s, size: 2128.9 KB)
val: Scanning /Users/duytan/orca/projects/PCB-Detect/PKU-Market-PCB(Data enhanced version)/train/labels.cache... 608 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 608/608 38.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 152/152 2.5it/s 1:00<0.4ss
                   all        608       3060      0.788      0.846      0.837      0.415
          Missing_hole        100        508      0.125      0.976      0.547      0.288
            Mouse_bite        100        504       0.92      0.842      0.889      0.415
          Open_circuit        104        524      0.919      0.864      0.896      0.465
                 Short        100        496      0.944

,images,imgsz,mAP50,mAP50_95,precision,recall,elapsed_seconds
model,,,,,,,
B0 (YOLOv8s),608,1024,0.8375,0.4150,0.7883,0.8456,64.0868
B1 (YOLOv8s-P2),608,1024,0.8793,0.4159,0.8338,0.8276,69.8221


model,B0 (YOLOv8s),B1 (YOLOv8s-P2)
class,,
Missing_hole,0.9764,0.9272
Mouse_bite,0.8419,0.8673
Open_circuit,0.8640,0.8263
Short,0.8162,0.8710
Spur,0.6640,0.5820
Spurious_copper,0.9114,0.8920


: 